# 11: MLP cfg03 seed stability

Rol: evaluar estabilidad de la mejor MLP tabular encontrada: MLP_tabular_cfg_03.

Este notebook no hace una busqueda grande. Repite la misma configuracion con seeds fijas y validacion artificial para ver si el buen MAE fue estable o una corrida afortunada. No usa test oficial.


In [10]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CMAPSSData").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [11]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)
from src.fd001_experiment_utils import (
    FlexibleMLPRegressor,
    add_bin_metric_columns,
    metric_row_from_predictions,
    selection_sort,
)


In [12]:
CFG03 = {
    "hidden_layers": [128, 64, 32],
    "dropout": 0.1,
    "weight_decay": 1e-4,
    "lr": 1e-3,
    "batch_size": 256,
    "max_epochs": 250,
    "patience": 25,
}
SEEDS = [0, 1, 2, 3, 4]
STABILITY_PATH = RESULTS_DIR / "fd001_mlp_cfg03_seed_stability.csv"
SUMMARY_PATH = RESULTS_DIR / "fd001_mlp_cfg03_seed_summary.csv"


In [13]:
prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=VALIDATION_RANDOM_STATE,
    max_rul=BASE_RUL_CAP,
    cut_ruls=CUT_RULS,
    window_size=BASE_WINDOW_SIZE,
)
preprocessing_summary(prepared)


,split,rows,units,features,target_mean,target_min,target_max
0,train,16561,80,119,86.958819,0.0,125.0
1,eval,99,20,119,79.393939,20.0,140.0


In [14]:
rows = []
prediction_tables = []
representation = f"temporal_w{BASE_WINDOW_SIZE}"
for seed in SEEDS:
    print(f"Entrenando MLP_tabular_cfg_03 seed={seed}")
    model = FlexibleMLPRegressor(**CFG03, random_state=seed)
    model.fit(prepared["X_train"], prepared["y_train"])
    preds = model.predict(prepared["X_eval"])
    pred_df = prediction_frame(
        prepared["eval_df"],
        preds,
        model_name="MLP_tabular_cfg_03",
        representation=representation,
    )
    pred_df["seed"] = seed
    prediction_tables.append(pred_df)
    row, _ = metric_row_from_predictions(
        pred_df,
        extra={
            "seed": seed,
            "model_name": "MLP_tabular_cfg_03",
            "representation": representation,
            "window_size": BASE_WINDOW_SIZE,
            "rul_cap": BASE_RUL_CAP,
            "sample_weight_scheme": "none",
            "hidden_layers": str(CFG03["hidden_layers"]),
            "dropout": CFG03["dropout"],
            "weight_decay": CFG03["weight_decay"],
            "lr": CFG03["lr"],
            "batch_size": CFG03["batch_size"],
            "epochs_run": len(model.history_),
        },
    )
    rows.append(row)

stability = selection_sort(pd.DataFrame(rows))
summary_rows = []
for metric in ["mae", "rmse", "r2", "cmapss_score", "cmapss_score_mean", "dangerous_error_pct"]:
    summary_rows.append({
        "model_name": "MLP_tabular_cfg_03",
        "representation": representation,
        "window_size": BASE_WINDOW_SIZE,
        "rul_cap": BASE_RUL_CAP,
        "metric": metric,
        "mean": stability[metric].mean(),
        "std": stability[metric].std(ddof=1),
    })
summary = pd.DataFrame(summary_rows)

stability.to_csv(STABILITY_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)
pd.concat(prediction_tables, ignore_index=True).to_csv(RESULTS_DIR / "fd001_mlp_cfg03_seed_predictions.csv", index=False)

display(stability)
display(summary)


Entrenando MLP_tabular_cfg_03 seed=0
Entrenando MLP_tabular_cfg_03 seed=1
Entrenando MLP_tabular_cfg_03 seed=2
Entrenando MLP_tabular_cfg_03 seed=3
Entrenando MLP_tabular_cfg_03 seed=4


,representation,model_name,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,seed,window_size,rul_cap,sample_weight_scheme,hidden_layers,dropout,weight_decay,lr,batch_size,epochs_run
0,temporal_w30,MLP_tabular_cfg_03,99,13.503936,18.296586,0.812083,522.540535,5.278187,11.111111,4.907073,0.0,9.523034,20.0,14.376378,35.0,19.506666,0.0,3,30,125,none,"[128, 64, 32]",0.1,0.0001,0.001,256,250
1,temporal_w30,MLP_tabular_cfg_03,99,13.774128,18.059696,0.816917,531.648509,5.370187,10.101010,4.754341,0.0,9.133415,10.0,16.673579,40.0,19.292615,0.0,1,30,125,none,"[128, 64, 32]",0.1,0.0001,0.001,256,250
2,temporal_w30,MLP_tabular_cfg_03,99,13.732640,17.993640,0.818254,576.420182,5.822426,13.131313,4.119559,0.0,9.720651,15.0,18.018053,50.0,18.522207,0.0,4,30,125,none,"[128, 64, 32]",0.1,0.0001,0.001,256,250
3,temporal_w30,MLP_tabular_cfg_03,99,14.036131,18.527586,0.807308,646.905090,6.534395,9.090909,4.189796,0.0,11.039830,10.0,18.352474,35.0,18.408588,0.0,0,30,125,none,"[128, 64, 32]",0.1,0.0001,0.001,256,250
4,temporal_w30,MLP_tabular_cfg_03,99,15.423398,20.126931,0.772605,716.971708,7.242138,13.131313,5.692428,0.0,9.651221,15.0,20.351147,50.0,20.846679,0.0,2,30,125,none,"[128, 64, 32]",0.1,0.0001,0.001,256,250


,model_name,representation,window_size,rul_cap,metric,mean,std
0,MLP_tabular_cfg_03,temporal_w30,30,125,mae,14.094047,0.766766
1,MLP_tabular_cfg_03,temporal_w30,30,125,rmse,18.600888,0.878695
2,MLP_tabular_cfg_03,temporal_w30,30,125,r2,0.805433,0.018853
3,MLP_tabular_cfg_03,temporal_w30,30,125,cmapss_score,598.897205,82.316903
4,MLP_tabular_cfg_03,temporal_w30,30,125,cmapss_score_mean,6.049467,0.831484
5,MLP_tabular_cfg_03,temporal_w30,30,125,dangerous_error_pct,11.313131,1.806924


El archivo results/fd001_mlp_metrics.csv se conserva como resultado historico de la busqueda inicial.
